# Task 2.2 — Source-intention benchmark and robust ensembles

This end-to-end benchmark distinguishes `DIRECT` from `JUDGEMENTAL` among memes already identified as sexist.

The experiment moves from inexpensive sparse baselines to multimodal experts: text and character TF-IDF, paper-guided physiological axes, frozen CLIP/DINOv2/ViT and MPNet embeddings, text-image-sensor fusion, cached VLM descriptions, fine-tuned transformer probabilities, and a regularized fusion MLP.

The audited benchmark evaluated **37 candidates** and retained **31 robust candidates**. The selected `geom_mean_top5` ensemble reaches **0.637 conditional macro-F1** and **0.469 F1 for `JUDGEMENTAL`**. When routed through Task 2.1, the diagnostic three-class macro-F1 is **0.485**.

The final report is the canonical source for these headline values and for the five selected members.

## 1. Setup and configuration

Paths and split logic come from `exist2026_meme_utils`. Heavy models are not retrained when compatible caches exist; cached fine-tuned transformer probabilities can be loaded as experts.

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
CACHE_DIR = PROJECT_ROOT / "outputs" / "cache"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
RUNS_DIR = PROJECT_ROOT / "outputs" / "runs"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

os.environ.setdefault("EXIST2026_PROJECT_ROOT", str(PROJECT_ROOT))

from exist2026_meme_utils import (
    attach_gold_and_soft,
    build_feature_frame,
    split_train_dev,
    text_inputs,
)
from task2_2_metric_focused_pipeline import (
    LABELS_BIN,
    NEG_LABEL,
    POS_LABEL,
    SEED,
    _align_matrix,
    _char_vectorizer,
    _union_vectorizer,
    _word_vectorizer,
    _vlm_enriched_text_inputs,
    add_paper_guided_sensor_axes,
    align_binary_proba,
    fast_metrics,
    normalize_binary,
    optimize_threshold,
    soft_gold_matrix,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 220)
np.random.seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CACHE_DIR exists:", CACHE_DIR.exists())


PROJECT_ROOT: /home/sortmon/LNR_CHG/EXIST2026_MEMES_ONLY_FINAL
CACHE_DIR exists: True


In [2]:
# Configuracion de alto nivel del notebook.
CONFIG = {
    "calib_size": 0.22,   # fraccion de train (no dev) reservada como conjunto de calibracion
    "seed": SEED,
    "n_folds_stacker": 5, # KFold para el stacker OOF
    "calib_dev_gap_max": 0.08,  # gap maximo |calib_macro - dev_macro| (overfit a calibracion)
    "min_dev_macro_f1": 0.50,   # macro-F1 minima sobre dev para que un candidato cuente
    "topk_simple_avg": 5,       # K para el ensamble 1
    "dirichlet_iters": 8000,    # iteraciones busqueda aleatoria ensamble 2
    "fallback_topk": 8,         # si el filtro deja menos de 3, tomar los K mejores por dev_macro_f1
    "include_finetuned_cache": True, # cargar probs cacheadas de XLM-R / mDeBERTa / DistilBERT
    "include_mlp_fusion": True,      # entrenar un MLP pequeno (PyTorch) si esta instalado
    "mlp_epochs": 25,
    "mlp_patience": 5,
}
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


  calib_size: 0.22
  seed: 42
  n_folds_stacker: 5
  calib_dev_gap_max: 0.08
  min_dev_macro_f1: 0.5
  topk_simple_avg: 5
  dirichlet_iters: 8000
  fallback_topk: 8
  include_finetuned_cache: True
  include_mlp_fusion: True
  mlp_epochs: 25
  mlp_patience: 5


## 2. Fit, calibration, and development partitions

`fit` trains each candidate, `calib` selects thresholds and ensemble settings, and `dev` audits internal stability. Because the stability filter consults development performance, the resulting development metrics are internal validation evidence, not a blind test estimate.

In [3]:
from sklearn.model_selection import train_test_split


def add_model_text(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["model_text"] = text_inputs(out).astype(str)
    return out


features = build_feature_frame(force=False)
train_all = features[features["split"].eq("training")].copy()
test_df = features[features["split"].eq("test")].copy().sort_values("id").reset_index(drop=True)

task22 = attach_gold_and_soft(train_all, "task2_2")
task22 = add_paper_guided_sensor_axes(task22)
test_df = add_paper_guided_sensor_axes(test_df)
task22_yes = task22[task22["gold"].isin(LABELS_BIN)].copy().reset_index(drop=True)

train_df_all, dev_df = split_train_dev(task22_yes, "task2_2")
stratify = train_df_all["lang"].astype(str) + "_" + train_df_all["gold"].astype(str)
fit_idx, calib_idx = train_test_split(
    np.arange(len(train_df_all)),
    test_size=CONFIG["calib_size"],
    random_state=CONFIG["seed"],
    stratify=stratify if stratify.value_counts().min() >= 2 else None,
)

fit_df = add_model_text(train_df_all.iloc[fit_idx].sort_values("id").reset_index(drop=True))
calib_df = add_model_text(train_df_all.iloc[calib_idx].sort_values("id").reset_index(drop=True))
dev_df = add_model_text(dev_df.sort_values("id").reset_index(drop=True))
test_df = add_model_text(test_df)

# Combined fit+calib used by stacker OOF.
combo_df = pd.concat([fit_df.assign(_fold_src="fit"), calib_df.assign(_fold_src="calib")], ignore_index=True)
combo_df = combo_df.sort_values("id").reset_index(drop=True)

y_fit = fit_df["gold"].astype(str).tolist()
y_calib = calib_df["gold"].astype(str).tolist()
y_dev = dev_df["gold"].astype(str).tolist()
y_combo = combo_df["gold"].astype(str).tolist()

y_calib_soft = soft_gold_matrix(calib_df)
y_dev_soft = soft_gold_matrix(dev_df)
y_combo_soft = soft_gold_matrix(combo_df)

print(f"fit:   n={len(fit_df):4d}  DIRECT={y_fit.count('DIRECT')}  JUDGEMENTAL={y_fit.count('JUDGEMENTAL')}")
print(f"calib: n={len(calib_df):4d}  DIRECT={y_calib.count('DIRECT')}  JUDGEMENTAL={y_calib.count('JUDGEMENTAL')}")
print(f"dev:   n={len(dev_df):4d}  DIRECT={y_dev.count('DIRECT')}  JUDGEMENTAL={y_dev.count('JUDGEMENTAL')}")
print(f"test:  n={len(test_df):4d}")


fit:   n=1111  DIRECT=834  JUDGEMENTAL=277
calib: n= 314  DIRECT=236  JUDGEMENTAL=78
dev:   n= 357  DIRECT=268  JUDGEMENTAL=89
test:  n=1053


## 3. Candidate representation and overfitting diagnostics

Every model emits probabilities aligned as `[JUDGEMENTAL, DIRECT]` for `fit`, `calib`, `dev`, and `test`. Thresholds are selected on calibration data; development metrics provide the internal stability audit.

In [4]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


@dataclass
class CandidatePred:
    name: str
    family: str
    description: str
    fit_probs: np.ndarray   # probs over fit set (training data, so naturally optimistic)
    calib_probs: np.ndarray
    dev_probs: np.ndarray
    test_probs: np.ndarray
    threshold: float = 0.5
    seconds: float = 0.0
    n_features: int | None = None
    extras: dict[str, Any] = field(default_factory=dict)


def _macro_f1(y_true: list[str], probs: np.ndarray, threshold: float) -> float:
    pred = [POS_LABEL if row[1] >= threshold else NEG_LABEL for row in normalize_binary(probs)]
    return float(f1_score(y_true, pred, labels=LABELS_BIN, average="macro", zero_division=0))


def _try_calibrated_svc(c: float) -> CalibratedClassifierCV:
    try:
        return CalibratedClassifierCV(
            estimator=LinearSVC(C=c, class_weight="balanced", random_state=CONFIG["seed"], dual="auto"),
            method="sigmoid", cv=3,
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=LinearSVC(C=c, class_weight="balanced", random_state=CONFIG["seed"]),
            method="sigmoid", cv=3,
        )


def predict_all_splits(pipe: Any, fit_x: Any, calib_x: Any, dev_x: Any, test_x: Any) -> dict[str, np.ndarray]:
    classes = pipe.named_steps["clf"].classes_ if hasattr(pipe, "named_steps") else pipe.classes_
    out = {}
    for tag, X in [("fit", fit_x), ("calib", calib_x), ("dev", dev_x), ("test", test_x)]:
        probs = pipe.predict_proba(X)
        out[tag] = align_binary_proba(classes, probs)
    return out


def register_candidate(
    pool: list[CandidatePred],
    name: str,
    family: str,
    description: str,
    probs_by_split: dict[str, np.ndarray],
    seconds: float = 0.0,
    n_features: int | None = None,
    extras: dict[str, Any] | None = None,
) -> CandidatePred:
    calib_probs = normalize_binary(probs_by_split["calib"])
    threshold, _ = optimize_threshold(y_calib, y_calib_soft, calib_probs)
    cand = CandidatePred(
        name=name,
        family=family,
        description=description,
        fit_probs=normalize_binary(probs_by_split["fit"]),
        calib_probs=calib_probs,
        dev_probs=normalize_binary(probs_by_split["dev"]),
        test_probs=normalize_binary(probs_by_split["test"]),
        threshold=float(threshold),
        seconds=float(seconds),
        n_features=n_features,
        extras=extras or {},
    )
    pool.append(cand)
    fit_macro = _macro_f1(y_fit, cand.fit_probs, cand.threshold)
    calib_macro = _macro_f1(y_calib, cand.calib_probs, cand.threshold)
    dev_macro = _macro_f1(y_dev, cand.dev_probs, cand.threshold)
    print(f"  + {name:42s} thr={cand.threshold:.2f}  fit={fit_macro:.3f}  calib={calib_macro:.3f}  dev={dev_macro:.3f}  ({seconds:.1f}s)")
    return cand


def _numeric_cols(frame: pd.DataFrame, prefixes: tuple[str, ...]) -> list[str]:
    return [
        col
        for col in frame.columns
        if col.startswith(prefixes) and pd.api.types.is_numeric_dtype(frame[col])
        and frame[col].notna().any()
    ]


candidates: list[CandidatePred] = []


## 4. Stage A — Classical text models

Sparse word and character models provide inexpensive, well-calibrated baselines.

In [5]:
text_specs = [
    ("Text_Word_LR_bal_C0.5", _word_vectorizer(90000),
     LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=2500, random_state=SEED),
     "Word TF-IDF + balanced LR"),
    ("Text_Word_LR_plain_C0.8", _word_vectorizer(90000),
     LogisticRegression(C=0.8, solver="liblinear", max_iter=2500, random_state=SEED),
     "Word TF-IDF + plain LR"),
    ("Text_WordChar_LR_bal_C0.3", _union_vectorizer(70000, 100000),
     LogisticRegression(C=0.3, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED),
     "Word+char TF-IDF + balanced LR"),
    ("Text_WordChar_LR_plain_C0.5", _union_vectorizer(70000, 100000),
     LogisticRegression(C=0.5, solver="liblinear", max_iter=3000, random_state=SEED),
     "Word+char TF-IDF + plain LR"),
    ("Text_WordChar_ComplementNB_a0.2", _union_vectorizer(65000, 90000),
     ComplementNB(alpha=0.2), "Word+char TF-IDF + ComplementNB"),
    ("Text_Word_MultinomialNB_a0.1", _word_vectorizer(90000),
     MultinomialNB(alpha=0.1), "Word TF-IDF + MultinomialNB"),
    ("Text_Char_LR_bal_C0.5", _char_vectorizer(120000),
     LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED),
     "Char-only TF-IDF + balanced LR"),
    ("Text_WordChar_LinearSVC_calib_C0.3", _union_vectorizer(70000, 100000),
     _try_calibrated_svc(0.3), "Word+char TF-IDF + calibrated LinearSVC"),
    ("Text_WordChar_SGD_logloss_bal", _union_vectorizer(70000, 90000),
     SGDClassifier(loss="log_loss", alpha=3e-5, class_weight="balanced", max_iter=2000, random_state=SEED),
     "Word+char TF-IDF + SGD log-loss"),
]

print("[Stage A] Texto clasico")
for name, vec, clf, desc in text_specs:
    t0 = time.time()
    pipe = Pipeline([("tfidf", vec), ("clf", clf)])
    pipe.fit(fit_df["model_text"], y_fit)
    probs = predict_all_splits(
        pipe, fit_df["model_text"], calib_df["model_text"], dev_df["model_text"], test_df["model_text"],
    )
    try:
        n_features = int(pipe.named_steps["tfidf"].transform(fit_df["model_text"].head(1)).shape[1])
    except Exception:
        n_features = None
    register_candidate(candidates, name, "text_classical", desc, probs, time.time() - t0, n_features)


[Stage A] Texto clasico
  + Text_Word_LR_bal_C0.5                      thr=0.45  fit=0.851  calib=0.546  dev=0.532  (0.1s)
  + Text_Word_LR_plain_C0.8                    thr=0.74  fit=0.837  calib=0.509  dev=0.538  (0.1s)


  + Text_WordChar_LR_bal_C0.3                  thr=0.45  fit=0.884  calib=0.580  dev=0.533  (0.7s)


  + Text_WordChar_LR_plain_C0.5                thr=0.68  fit=0.914  calib=0.570  dev=0.538  (0.8s)


  + Text_WordChar_ComplementNB_a0.2            thr=0.78  fit=0.876  calib=0.552  dev=0.589  (0.4s)
  + Text_Word_MultinomialNB_a0.1               thr=0.82  fit=0.858  calib=0.526  dev=0.527  (0.1s)


  + Text_Char_LR_bal_C0.5                      thr=0.47  fit=0.872  calib=0.597  dev=0.528  (0.6s)


  + Text_WordChar_LinearSVC_calib_C0.3         thr=0.74  fit=0.223  calib=0.468  dev=0.430  (0.4s)


  + Text_WordChar_SGD_logloss_bal              thr=0.56  fit=0.998  calib=0.510  dev=0.508  (0.4s)


## 5. Stage B — Text and physiological sensors

TF-IDF features are concatenated with paper-guided ET/HR/EEG axes and classified with logistic regression. This is the simplest joint content-and-observer branch.

In [6]:
sensor_cols = _numeric_cols(fit_df, ("sens_", "paper_"))
visual_cols = _numeric_cols(fit_df, ("vis_",))


def _column_text_numeric_pipeline(vectorizer, numeric_cols, clf, scale_numeric=True):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    pre = ColumnTransformer(
        [("text", vectorizer, "model_text"),
         ("numeric", Pipeline(num_steps), numeric_cols)],
        remainder="drop", sparse_threshold=0.35,
    )
    return Pipeline([("pre", pre), ("clf", clf)])


print("[Stage B] Texto + sensores")
if sensor_cols:
    fusion_specs = [
        ("Fusion_TextSensors_WordChar_LR", _union_vectorizer(65000, 90000),
         LogisticRegression(C=0.35, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED),
         "Word+char TF-IDF + paper ET/HR/EEG axes"),
        ("Fusion_TextSensors_Word_LR", _word_vectorizer(90000),
         LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED),
         "Word TF-IDF + paper ET/HR/EEG axes"),
    ]
    for name, vec, clf, desc in fusion_specs:
        t0 = time.time()
        pipe = _column_text_numeric_pipeline(vec, sensor_cols, clf, scale_numeric=True)
        pipe.fit(fit_df, y_fit)
        probs = predict_all_splits(pipe, fit_df, calib_df, dev_df, test_df)
        register_candidate(candidates, name, "text_plus_sensors", desc, probs, time.time() - t0, len(sensor_cols))
else:
    print("  Sin columnas sensoriales disponibles. Saltando.")


[Stage B] Texto + sensores


  + Fusion_TextSensors_WordChar_LR             thr=0.55  fit=0.782  calib=0.533  dev=0.508  (3.4s)
  + Fusion_TextSensors_Word_LR                 thr=0.54  fit=0.766  calib=0.530  dev=0.491  (0.2s)


## 6. Stage C — Sensor-only models

Sensor-only candidates measure how much conditional intention signal can be attributed to physiological response rather than meme content. Large calibration-only gains are treated as a warning sign.

In [7]:
print("[Stage C] Sensores solos")


def _fit_numeric_pipe(fit_df, calib_df, dev_df, test_df, cols, clf, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("clf", clf))
    pipe = Pipeline(steps)
    pipe.fit(fit_df[cols], y_fit)
    return predict_all_splits(pipe, fit_df[cols], calib_df[cols], dev_df[cols], test_df[cols])


sensor_specs = [
    ("Sensors_paperAll_HGB", ("paper_",), HistGradientBoostingClassifier(max_iter=90, learning_rate=0.055, l2_regularization=0.05, random_state=SEED), False,
     "Paper axes (ET+HR+EEG) + HGB"),
    ("Sensors_paperET_HR_LR", ("paper_et_", "paper_hr_"), LogisticRegression(C=0.35, class_weight="balanced", solver="liblinear", max_iter=1600, random_state=SEED), True,
     "Paper ET+HR axes + balanced LR"),
    ("Sensors_paperEEG_HGB", ("paper_eeg_",), HistGradientBoostingClassifier(max_iter=80, learning_rate=0.06, l2_regularization=0.07, random_state=SEED), False,
     "Paper EEG axes + HGB"),
    ("Sensors_AllRaw_ExtraTrees", ("sens_",), ExtraTreesClassifier(n_estimators=260, min_samples_leaf=3, max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1), False,
     "Raw sens_* + ExtraTrees"),
]
for name, prefixes, clf, scale, desc in sensor_specs:
    cols = _numeric_cols(fit_df, prefixes)
    if not cols:
        print(f"  - {name}: sin columnas {prefixes}, saltando.")
        continue
    t0 = time.time()
    probs = _fit_numeric_pipe(fit_df, calib_df, dev_df, test_df, cols, clf, scale=scale)
    register_candidate(candidates, name, "sensors_only", desc, probs, time.time() - t0, len(cols))


[Stage C] Sensores solos


  + Sensors_paperAll_HGB                       thr=0.82  fit=0.963  calib=0.531  dev=0.506  (0.2s)
  + Sensors_paperET_HR_LR                      thr=0.46  fit=0.576  calib=0.564  dev=0.544  (0.0s)


  + Sensors_paperEEG_HGB                       thr=0.70  fit=0.992  calib=0.525  dev=0.495  (0.2s)


  + Sensors_AllRaw_ExtraTrees                  thr=0.62  fit=1.000  calib=0.561  dev=0.552  (0.3s)


## 7. Stage D — Frozen image embeddings

CLIP, DINOv2, and ViT provide image-only representations without OCR, testing whether the visual channel independently separates `DIRECT` from `JUDGEMENTAL`.

In [8]:
image_caches = {
    "CLIPImage":    CACHE_DIR / "all_memes_clip_openai_clip-vit-base-patch32_5037.npz",
    "DINOv2Image":  CACHE_DIR / "all_memes_imagedense_dinov2small_facebook_dinov2_small_5037.npz",
    "ViTImage":     CACHE_DIR / "all_memes_imagedense_vitbasein21k_google_vit_base_patch16_224_in21k_5037.npz",
}


def _load_embedding_blocks(path: Path) -> dict[str, np.ndarray] | None:
    ids_fit = fit_df["id"].astype(str).tolist()
    ids_calib = calib_df["id"].astype(str).tolist()
    ids_dev = dev_df["id"].astype(str).tolist()
    ids_test = test_df["id"].astype(str).tolist()
    parts = {}
    for tag, ids in [("fit", ids_fit), ("calib", ids_calib), ("dev", ids_dev), ("test", ids_test)]:
        block = _align_matrix(path, ids)
        if block is None:
            return None
        parts[tag] = block
    return parts


def _fit_lr_on_embedding(parts: dict[str, np.ndarray], y_fit: list[str], c: float = 0.4) -> dict[str, np.ndarray]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=c, class_weight="balanced", solver="liblinear", max_iter=1600, random_state=SEED)),
    ])
    pipe.fit(parts["fit"], y_fit)
    classes = pipe.named_steps["clf"].classes_
    out = {tag: align_binary_proba(classes, pipe.predict_proba(parts[tag])) for tag in parts}
    return out


print("[Stage D] Imagen sola")
image_parts_cache: dict[str, dict[str, np.ndarray]] = {}
for short, path in image_caches.items():
    parts = _load_embedding_blocks(path)
    if parts is None:
        print(f"  - {short}: cache no disponible ({path.name}), saltando.")
        continue
    image_parts_cache[short] = parts
    t0 = time.time()
    probs = _fit_lr_on_embedding(parts, y_fit)
    register_candidate(candidates, f"Image_{short}_LR", "image_embedding", f"{short} embedding + balanced LR",
                       probs, time.time() - t0, parts["fit"].shape[1])

# Imagen handcrafted como sanity check.
if visual_cols:
    t0 = time.time()
    probs = _fit_numeric_pipe(
        fit_df, calib_df, dev_df, test_df, visual_cols,
        HistGradientBoostingClassifier(max_iter=90, learning_rate=0.055, l2_regularization=0.05, random_state=SEED),
        scale=False,
    )
    register_candidate(candidates, "Image_visualNumeric_HGB", "image_numeric",
                       "Handcrafted vis_* features + HGB", probs, time.time() - t0, len(visual_cols))


[Stage D] Imagen sola


  + Image_CLIPImage_LR                         thr=0.58  fit=0.864  calib=0.562  dev=0.559  (0.1s)
  + Image_DINOv2Image_LR                       thr=0.38  fit=0.811  calib=0.514  dev=0.434  (0.0s)


  + Image_ViTImage_LR                          thr=0.60  fit=0.969  calib=0.513  dev=0.515  (0.1s)


  + Image_visualNumeric_HGB                    thr=0.80  fit=0.996  calib=0.528  dev=0.492  (0.3s)


## 8. Stage E — Dense text embeddings

Multilingual MPNet captures semantic information beyond exact OCR tokens without requiring expensive end-to-end fine-tuning.

In [9]:
text_emb_caches = {
    "MPNetText": CACHE_DIR / "all_memes_sentence_sentence-transformers_paraphrase-multilingual-mpnet-base-v2_5037.npz",
}

print("[Stage E] Texto embedding")
text_parts_cache: dict[str, dict[str, np.ndarray]] = {}
for short, path in text_emb_caches.items():
    parts = _load_embedding_blocks(path)
    if parts is None:
        print(f"  - {short}: cache no disponible, saltando.")
        continue
    text_parts_cache[short] = parts
    t0 = time.time()
    probs = _fit_lr_on_embedding(parts, y_fit, c=0.45)
    register_candidate(candidates, f"Text_{short}_LR", "text_embedding",
                       f"{short} embedding + balanced LR", probs, time.time() - t0, parts["fit"].shape[1])


[Stage E] Texto embedding


  + Text_MPNetText_LR                          thr=0.22  fit=0.916  calib=0.553  dev=0.512  (0.2s)


## 9. Stage F — Text-image late fusion

Dense text and image embeddings are concatenated and classified linearly, providing a reproducible late-fusion counterpart to a jointly fine-tuned text-vision model.

In [10]:
print("[Stage F] Texto + imagen embedding")
fusion_specs = [
    ("Fusion_MPNet_CLIP_LR", ["MPNetText", "CLIPImage"], "MPNet text + CLIP image"),
    ("Fusion_MPNet_DINO_LR", ["MPNetText", "DINOv2Image"], "MPNet text + DINOv2 image"),
    ("Fusion_MPNet_AllImages_LR", ["MPNetText", "CLIPImage", "DINOv2Image", "ViTImage"], "MPNet + CLIP+DINO+ViT"),
]
all_parts = {**text_parts_cache, **image_parts_cache}
for name, keys, desc in fusion_specs:
    if not all(k in all_parts for k in keys):
        print(f"  - {name}: faltan embeddings ({keys}), saltando.")
        continue
    t0 = time.time()
    parts = {
        tag: np.hstack([all_parts[k][tag] for k in keys])
        for tag in ("fit", "calib", "dev", "test")
    }
    probs = _fit_lr_on_embedding(parts, y_fit, c=0.4)
    register_candidate(candidates, name, "fusion_text_image", desc, probs, time.time() - t0, parts["fit"].shape[1])


[Stage F] Texto + imagen embedding


  + Fusion_MPNet_CLIP_LR                       thr=0.20  fit=0.999  calib=0.558  dev=0.528  (0.3s)


  + Fusion_MPNet_DINO_LR                       thr=0.35  fit=0.999  calib=0.545  dev=0.534  (0.3s)


  + Fusion_MPNet_AllImages_LR                  thr=0.82  fit=0.999  calib=0.528  dev=0.489  (0.8s)


## 10. Stage G — Text, image, and sensor fusion

Dense content embeddings and interpretable physiological axes are concatenated in a single classifier, approximating a full multimodal system with a simpler training regime.

In [11]:
def _stack_sensors_parts():
    if not sensor_cols:
        return None
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    fit_x = imputer.fit_transform(fit_df[sensor_cols])
    fit_x = scaler.fit_transform(fit_x)
    parts = {"fit": fit_x}
    for tag, frame in [("calib", calib_df), ("dev", dev_df), ("test", test_df)]:
        x = imputer.transform(frame[sensor_cols])
        parts[tag] = scaler.transform(x)
    return parts


sensors_parts = _stack_sensors_parts()
print("[Stage G] Multimodal + sensores")
mm_specs = [
    ("Full_MPNet_CLIP_Sensors_LR", ["MPNetText", "CLIPImage"], True, "MPNet + CLIP + paper sensors"),
    ("Full_MPNet_AllImg_Sensors_LR", ["MPNetText", "CLIPImage", "DINOv2Image", "ViTImage"], True, "MPNet + CLIP+DINO+ViT + paper sensors"),
]
for name, keys, with_sensors, desc in mm_specs:
    if not all(k in all_parts for k in keys):
        print(f"  - {name}: faltan embeddings, saltando.")
        continue
    if with_sensors and sensors_parts is None:
        print(f"  - {name}: sin sensores, saltando.")
        continue
    t0 = time.time()
    parts = {}
    for tag in ("fit", "calib", "dev", "test"):
        blocks = [all_parts[k][tag] for k in keys]
        if with_sensors:
            blocks.append(sensors_parts[tag])
        parts[tag] = np.hstack(blocks)
    probs = _fit_lr_on_embedding(parts, y_fit, c=0.35)
    register_candidate(candidates, name, "fusion_text_image_sensors", desc, probs, time.time() - t0, parts["fit"].shape[1])


[Stage G] Multimodal + sensores


  + Full_MPNet_CLIP_Sensors_LR                 thr=0.68  fit=1.000  calib=0.530  dev=0.552  (0.4s)


  + Full_MPNet_AllImg_Sensors_LR               thr=0.70  fit=1.000  calib=0.547  dev=0.518  (0.7s)


## 11. Stage H — VLM-enriched text

When `outputs/cache/task2_2_vlm_reasoning.csv` is available, cached visual-language descriptions are appended to OCR. Coverage must be audited across every target split before this branch is used for deployment.

In [ ]:
vlm_cache = CACHE_DIR / "task2_2_vlm_reasoning.csv"
if vlm_cache.exists():
    print("[Stage H] VLM-enriched text disponible")
    fit_text_vlm = _vlm_enriched_text_inputs(fit_df).astype(str)
    calib_text_vlm = _vlm_enriched_text_inputs(calib_df).astype(str)
    dev_text_vlm = _vlm_enriched_text_inputs(dev_df).astype(str)
    test_text_vlm = _vlm_enriched_text_inputs(test_df).astype(str)
    coverage = {
        "fit_has_vlm": int((fit_text_vlm != fit_df["model_text"].astype(str)).sum()),
        "calib_has_vlm": int((calib_text_vlm != calib_df["model_text"].astype(str)).sum()),
        "dev_has_vlm": int((dev_text_vlm != dev_df["model_text"].astype(str)).sum()),
        "test_has_vlm": int((test_text_vlm != test_df["model_text"].astype(str)).sum()),
    }
    print("  cobertura:", coverage)
    if coverage["test_has_vlm"] == 0:
        print("  WARNING: cobertura VLM de test nula; estos candidatos no deben justificarse como sistema final desplegado.")
    specs = [
        ("VLM_WordChar_LR", _union_vectorizer(70000, 100000),
         LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=3000, random_state=SEED),
         "OCR + Qwen2.5-VL reasoning, word+char TF-IDF + LR"),
        ("VLM_WordChar_ComplementNB", _union_vectorizer(65000, 90000),
         ComplementNB(alpha=0.2), "OCR + Qwen2.5-VL reasoning + ComplementNB"),
    ]
    for name, vec, clf, desc in specs:
        t0 = time.time()
        pipe = Pipeline([("tfidf", vec), ("clf", clf)])
        pipe.fit(fit_text_vlm, y_fit)
        probs = predict_all_splits(pipe, fit_text_vlm, calib_text_vlm, dev_text_vlm, test_text_vlm)
        register_candidate(candidates, name, "vlm_reasoning_text", desc, probs, time.time() - t0, None,
                           extras={"coverage": coverage})
else:
    print("[Stage H] Sin cache VLM, se omite. (Generar con scripts/generate_ollama_vlm_reasoning.py si se quiere usar.)")


## 12. Stage I — Cached fine-tuned transformers

Previously generated probability caches are loaded only when their development IDs align completely. Experts without genuine out-of-fold training probabilities remain voting members and are excluded from OOF stacking.

In [13]:
import glob


def _load_v2_finetuned_cache(path: Path) -> dict[str, np.ndarray] | None:
    # v2 cache has fit/calib/dev/test_ids + corresponding _probs.
    data = np.load(path, allow_pickle=True)
    required = {"fit_ids", "calib_ids", "dev_ids", "test_ids",
                "fit_probs", "calib_probs", "dev_probs", "test_probs"}
    if not required <= set(data.keys()):
        return None

    def _lookup(ids_arr):
        return {str(x): i for i, x in enumerate(ids_arr)}

    fit_lkp = _lookup(data["fit_ids"])
    cal_lkp = _lookup(data["calib_ids"])
    dev_lkp = _lookup(data["dev_ids"])
    tst_lkp = _lookup(data["test_ids"])

    fit_target = [str(x) for x in fit_df["id"].tolist()]
    cal_target = [str(x) for x in calib_df["id"].tolist()]
    dev_target = [str(x) for x in dev_df["id"].tolist()]
    tst_target = [str(x) for x in test_df["id"].tolist()]

    for target, lkp in [(fit_target, fit_lkp), (cal_target, cal_lkp), (dev_target, dev_lkp), (tst_target, tst_lkp)]:
        if any(item not in lkp for item in target):
            return None

    return {
        "fit":   normalize_binary(np.stack([np.asarray(data["fit_probs"][fit_lkp[i]], dtype=float) for i in fit_target])),
        "calib": normalize_binary(np.stack([np.asarray(data["calib_probs"][cal_lkp[i]], dtype=float) for i in cal_target])),
        "dev":   normalize_binary(np.stack([np.asarray(data["dev_probs"][dev_lkp[i]], dtype=float) for i in dev_target])),
        "test":  normalize_binary(np.stack([np.asarray(data["test_probs"][tst_lkp[i]], dtype=float) for i in tst_target])),
    }


def _load_legacy_finetuned_cache(path: Path) -> dict[str, np.ndarray] | None:
    # Legacy cache only has dev/test. Fill fit/calib with mid-prior derived from dev
    # (NOT used for threshold tuning of the candidate; threshold is fixed to 0.5).
    data = np.load(path, allow_pickle=True)
    if not {"dev_ids", "dev_probs", "test_ids", "test_probs"} <= set(data.keys()):
        return None
    dev_lkp = {str(x): i for i, x in enumerate(data["dev_ids"])}
    tst_lkp = {str(x): i for i, x in enumerate(data["test_ids"])}
    dev_target = [str(x) for x in dev_df["id"].tolist()]
    tst_target = [str(x) for x in test_df["id"].tolist()]
    if any(i not in dev_lkp for i in dev_target):
        return None
    if any(i not in tst_lkp for i in tst_target):
        return None
    dev_probs = normalize_binary(np.stack([np.asarray(data["dev_probs"][dev_lkp[i]], dtype=float) for i in dev_target]))
    tst_probs = normalize_binary(np.stack([np.asarray(data["test_probs"][tst_lkp[i]], dtype=float) for i in tst_target]))
    # Use a prior-only matrix (0.5/0.5) so threshold tuning on calib is neutral.
    fit_probs = np.tile([[0.5, 0.5]], (len(fit_df), 1))
    cal_probs = np.tile([[0.5, 0.5]], (len(calib_df), 1))
    return {"fit": fit_probs, "calib": cal_probs, "dev": dev_probs, "test": tst_probs}


if CONFIG["include_finetuned_cache"]:
    print("[Stage I] Transformers fine-tuned cacheados")
    # 1) Prefer v2 caches (have proper fit+calib+dev+test probs from notebook split).
    v2_paths = sorted(glob.glob(str(CACHE_DIR / "task22_transformer_v2_*.npz")))
    legacy_paths = sorted(glob.glob(str(CACHE_DIR / "task22_transformer_probs_*.npz")))
    loaded_v2 = 0
    for path in v2_paths:
        name = Path(path).stem.replace("task22_transformer_v2_", "TransformerV2_")
        probs = _load_v2_finetuned_cache(Path(path))
        if probs is None:
            continue
        loaded_v2 += 1
        register_candidate(candidates, name, "transformer_finetuned_v2",
                           f"Retrained on fit/calib/dev/test: {Path(path).name}", probs, 0.0)
        if loaded_v2 >= 10:
            break
    print(f"  v2 caches cargados: {loaded_v2}")

    if loaded_v2 == 0:
        # Fallback to legacy caches (dev/test only) but mark as risky.
        loaded_legacy = 0
        for path in legacy_paths:
            name = Path(path).stem.replace("task22_transformer_probs_", "Transformer_")
            probs = _load_legacy_finetuned_cache(Path(path))
            if probs is None:
                continue
            loaded_legacy += 1
            register_candidate(candidates, name, "transformer_finetuned_legacy",
                               f"Legacy probabilities (dev+test only): {Path(path).name}", probs, 0.0)
            if loaded_legacy >= 6:
                break
        print(f"  legacy caches cargados (fallback): {loaded_legacy}")
        if loaded_legacy == 0:
            print("  (ninguna cache cubre el dev actual)")


[Stage I] Transformers fine-tuned cacheados


  + TransformerV2_cardiffnlp_twitter-xlm-roberta-base-sentiment_ocr_lang_e3_L192 thr=0.64  fit=0.702  calib=0.540  dev=0.529  (0.0s)


  + TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L192_lr3e-05 thr=0.49  fit=0.787  calib=0.585  dev=0.560  (0.0s)


  + TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L256_lr3e-05_soft thr=0.38  fit=0.610  calib=0.580  dev=0.592  (0.0s)
  + TransformerV2_microsoft_mdeberta-v3-base_ocr_lang_e3_L192_lr1.5e-05 thr=0.48  fit=0.601  calib=0.562  dev=0.544  (0.0s)


  + TransformerV2_microsoft_mdeberta-v3-base_ocr_lang_e4_L256_lr1e-05_soft thr=0.38  fit=0.657  calib=0.554  dev=0.555  (0.0s)


  + TransformerV2_xlm-roberta-base_ocr_lang_e4_L192 thr=0.49  fit=0.628  calib=0.590  dev=0.558  (0.0s)


  + TransformerV2_xlm-roberta-base_ocr_lang_e4_L256_lr1.5e-05_soft thr=0.34  fit=0.562  calib=0.563  dev=0.531  (0.0s)


  + TransformerV2_xlm-roberta-base_qwen_rich_e4_L256_lr1.5e-05_soft thr=0.36  fit=0.653  calib=0.612  dev=0.631  (0.0s)
  + TransformerV2_xlm-roberta-large_ocr_lang_e3_L192_lr1e-05 thr=0.51  fit=0.572  calib=0.554  dev=0.553  (0.0s)
  v2 caches cargados: 9


## 13. Stage J — Regularized fusion MLP

A small MLP combines MPNet, CLIP, and sensor features with dropout, weight decay, and early stopping.

In [14]:
if CONFIG["include_mlp_fusion"]:
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset
        has_torch = True
    except Exception as exc:
        print(f"[Stage J] torch no disponible: {exc}")
        has_torch = False
else:
    has_torch = False


if has_torch and ("MPNetText" in all_parts) and ("CLIPImage" in all_parts):
    print("[Stage J] MLP fusion regularizado")
    keys = ["MPNetText", "CLIPImage"]
    parts = {
        tag: np.hstack([all_parts[k][tag] for k in keys] + ([sensors_parts[tag]] if sensors_parts is not None else []))
        for tag in ("fit", "calib", "dev", "test")
    }
    in_dim = parts["fit"].shape[1]
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    y_idx = {"JUDGEMENTAL": 0, "DIRECT": 1}
    y_fit_t = torch.tensor([y_idx[v] for v in y_fit], dtype=torch.long, device=device)
    y_calib_t = torch.tensor([y_idx[v] for v in y_calib], dtype=torch.long, device=device)

    class RegMLP(nn.Module):
        def __init__(self, in_dim, hidden=256, p_drop=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(p_drop),
                nn.Linear(hidden, hidden // 2), nn.LayerNorm(hidden // 2), nn.GELU(), nn.Dropout(p_drop),
                nn.Linear(hidden // 2, 2),
            )
        def forward(self, x):
            return self.net(x)

    # Class weights to deal with imbalance.
    counts = pd.Series(y_fit).value_counts()
    cw = torch.tensor([
        len(y_fit) / max(1, counts.get("JUDGEMENTAL", 1)) * 0.5,
        len(y_fit) / max(1, counts.get("DIRECT", 1)) * 0.5,
    ], dtype=torch.float32, device=device)

    fit_x_t = torch.tensor(parts["fit"], dtype=torch.float32, device=device)
    calib_x_t = torch.tensor(parts["calib"], dtype=torch.float32, device=device)
    dev_x_t = torch.tensor(parts["dev"], dtype=torch.float32, device=device)
    test_x_t = torch.tensor(parts["test"], dtype=torch.float32, device=device)

    model = RegMLP(in_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["mlp_epochs"])
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    best_macro = -math.inf
    best_state = None
    patience = 0
    t0 = time.time()
    loader = DataLoader(TensorDataset(fit_x_t, y_fit_t), batch_size=64, shuffle=True)
    for epoch in range(CONFIG["mlp_epochs"]):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            calib_logits = model(calib_x_t)
            calib_probs = torch.softmax(calib_logits, dim=1).cpu().numpy()
        macro = _macro_f1(y_calib, calib_probs, 0.5)
        if macro > best_macro + 1e-4:
            best_macro = macro
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= CONFIG["mlp_patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probs = {}
        for tag, X in [("fit", fit_x_t), ("calib", calib_x_t), ("dev", dev_x_t), ("test", test_x_t)]:
            probs[tag] = torch.softmax(model(X), dim=1).cpu().numpy()
    register_candidate(candidates, "MLP_MPNet_CLIP_Sensors", "mlp_fusion_regularized",
                       "MLP small + dropout 0.3 + wd 0.01 + early stop", probs, time.time() - t0, in_dim)
else:
    print("[Stage J] Skipped (sin torch o sin embeddings necesarios).")


[Stage J] MLP fusion regularizado


  + MLP_MPNet_CLIP_Sensors                     thr=0.50  fit=0.898  calib=0.568  dev=0.584  (0.4s)


## 14. Candidate audit and stability filter

Each candidate is audited using fit, calibration, and development macro-F1 plus fit–development and calibration–development gaps. The retained pool requires development macro-F1 of at least 0.50 and an absolute calibration–development gap no larger than 0.08. Fit–development differences remain diagnostic because sparse models naturally fit training data strongly.

In [15]:
def _full_metrics(y_true: list[str], probs: np.ndarray, threshold: float) -> dict[str, float]:
    pred = [POS_LABEL if row[1] >= threshold else NEG_LABEL for row in normalize_binary(probs)]
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, pred, labels=LABELS_BIN, zero_division=0
    )
    try:
        auc = float(roc_auc_score([1 if y == POS_LABEL else 0 for y in y_true], normalize_binary(probs)[:, 1]))
    except Exception:
        auc = math.nan
    return {
        "macro_f1": float(f1_score(y_true, pred, labels=LABELS_BIN, average="macro", zero_division=0)),
        "judgemental_f1": float(f1[0]),
        "direct_f1": float(f1[1]),
        "judgemental_recall": float(recall[0]),
        "direct_recall": float(recall[1]),
        "auc_direct": auc,
    }


rows = []
for cand in candidates:
    fit_m = _full_metrics(y_fit, cand.fit_probs, cand.threshold)
    calib_m = _full_metrics(y_calib, cand.calib_probs, cand.threshold)
    dev_m = _full_metrics(y_dev, cand.dev_probs, cand.threshold)
    rows.append({
        "candidate": cand.name,
        "family": cand.family,
        "threshold": cand.threshold,
        "seconds": cand.seconds,
        "n_features": cand.n_features if cand.n_features is not None else -1,
        "fit_macro_f1": fit_m["macro_f1"],
        "calib_macro_f1": calib_m["macro_f1"],
        "dev_macro_f1": dev_m["macro_f1"],
        "dev_judgemental_f1": dev_m["judgemental_f1"],
        "dev_direct_f1": dev_m["direct_f1"],
        "dev_auc_direct": dev_m["auc_direct"],
        "gap_fit_minus_dev": fit_m["macro_f1"] - dev_m["macro_f1"],
        "gap_calib_minus_dev": calib_m["macro_f1"] - dev_m["macro_f1"],
    })

metrics_df = pd.DataFrame(rows).sort_values("dev_macro_f1", ascending=False).reset_index(drop=True)
metrics_df.to_csv(TABLES_DIR / "task2_2_master_candidate_metrics.csv", index=False)

robust_mask = (
    (metrics_df["dev_macro_f1"] >= CONFIG["min_dev_macro_f1"])
    & (metrics_df["gap_calib_minus_dev"].abs() <= CONFIG["calib_dev_gap_max"])
)
robust_df = metrics_df[robust_mask].copy()
print(f"\nCandidatos totales:   {len(metrics_df)}")
print(f"Candidatos robustos:  {len(robust_df)} (dev_macro_f1>={CONFIG['min_dev_macro_f1']}, |calib-dev|<={CONFIG['calib_dev_gap_max']})")

if len(robust_df) < 3:
    print(f"  Fallback: tomando top {CONFIG['fallback_topk']} por dev_macro_f1.")
    robust_df = metrics_df.head(CONFIG["fallback_topk"]).copy()

display(metrics_df.head(25))



Candidatos totales:   37
Candidatos robustos:  31 (dev_macro_f1>=0.5, |calib-dev|<=0.08)


,candidate,family,threshold,seconds,n_features,fit_macro_f1,calib_macro_f1,dev_macro_f1,dev_judgemental_f1,dev_direct_f1,dev_auc_direct,gap_fit_minus_dev,gap_calib_minus_dev
0,VLM_WordChar_LR,vlm_reasoning_text,0.500625,0.881455,-1,0.911692,0.588340,0.641413,0.476684,0.806142,0.708620,0.270279,-0.053073
1,TransformerV2_xlm-roberta-base_qwen_rich_e4_L256_lr1.5e-05_soft,transformer_finetuned_v2,0.362500,0.000000,-1,0.652756,0.612143,0.630937,0.525490,0.736383,0.724090,0.021819,-0.018794
2,VLM_WordChar_ComplementNB,vlm_reasoning_text,0.728125,0.465843,-1,0.878404,0.594075,0.629542,0.467662,0.791423,0.693653,0.248862,-0.035467
3,TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L256_lr3e-05_soft,transformer_finetuned_v2,0.378750,0.000000,-1,0.610129,0.579771,0.592440,0.439462,0.745418,0.656025,0.017689,-0.012668
4,Text_WordChar_ComplementNB_a0.2,text_classical,0.785000,0.377684,32065,0.875889,0.552079,0.588513,0.422535,0.754491,0.608586,0.287376,-0.036434
5,MLP_MPNet_CLIP_Sensors,mlp_fusion_regularized,0.500625,0.419949,1564,0.898045,0.568262,0.583959,0.411483,0.756436,0.612821,0.314086,-0.015697
6,TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L192_lr3e-05,transformer_finetuned_v2,0.492500,0.000000,-1,0.787206,0.584656,0.559746,0.387097,0.732394,0.617894,0.227460,0.024911
7,Image_CLIPImage_LR,image_embedding,0.581875,0.082814,512,0.863928,0.562408,0.558647,0.370732,0.746562,0.585569,0.305281,0.003762
8,TransformerV2_xlm-roberta-base_ocr_lang_e4_L192,transformer_finetuned_v2,0.492500,0.000000,-1,0.627986,0.589946,0.557653,0.392857,0.722449,0.592718,0.070333,0.032293
9,TransformerV2_microsoft_mdeberta-v3-base_ocr_lang_e4_L256_lr1e-05_soft,transformer_finetuned_v2,0.378750,0.000000,-1,0.656636,0.554304,0.555474,0.391111,0.719836,0.585716,0.101162,-0.001170


## 15. Ensemble strategy pool

The benchmark evaluates soft voting, heuristic weighting, geometric means, rank averaging, regularized Dirichlet weight search, and OOF stackers. Selection is performed after the documented stability audit.

In [16]:
def _candidates_by_names(names: list[str]) -> list[CandidatePred]:
    lookup = {c.name: c for c in candidates}
    return [lookup[name] for name in names if name in lookup]


def _ensemble_probs(cands: list[CandidatePred], weights: np.ndarray, tag: str) -> np.ndarray:
    stack = np.stack([getattr(c, f"{tag}_probs") for c in cands], axis=0)
    weights = np.asarray(weights, dtype=float)
    weights = weights / max(1e-9, weights.sum())
    return (weights[:, None, None] * stack).sum(axis=0)


def _geom_mean_probs(cands: list[CandidatePred], tag: str) -> np.ndarray:
    stack = np.stack([getattr(c, f"{tag}_probs") for c in cands], axis=0)
    log_avg = np.log(np.clip(stack, 1e-9, 1.0)).mean(axis=0)
    out = np.exp(log_avg)
    return out / out.sum(axis=1, keepdims=True)


def _rank_average_probs(cands: list[CandidatePred], tag: str) -> np.ndarray:
    # Average the per-candidate ranks of P(DIRECT). Output is a calibrated prob in [0,1].
    direct_probs = np.stack([getattr(c, f"{tag}_probs")[:, 1] for c in cands], axis=0)
    ranks = np.argsort(np.argsort(direct_probs, axis=1), axis=1) / max(1, direct_probs.shape[1] - 1)
    avg = ranks.mean(axis=0)
    return np.stack([1.0 - avg, avg], axis=1)


def _eval_ensemble_on_calib(calib_probs: np.ndarray) -> tuple[float, float]:
    thr, m = optimize_threshold(y_calib, y_calib_soft, calib_probs)
    return float(thr), float(m["macro_f1"])


robust_names = robust_df.sort_values("dev_macro_f1", ascending=False)["candidate"].tolist()
robust_cands = _candidates_by_names(robust_names)
print(f"Pool robusto: {len(robust_cands)} candidatos")

ensemble_records: list[dict[str, Any]] = []


def _register_ensemble(name: str, strategy: str, members: list[CandidatePred], weights, calib_probs, dev_probs, test_probs):
    threshold, calib_macro = _eval_ensemble_on_calib(calib_probs)
    dev_m = _full_metrics(y_dev, dev_probs, threshold)
    record = {
        "name": name,
        "strategy": strategy,
        "members": [c.name for c in members],
        "weights": (weights.tolist() if isinstance(weights, np.ndarray) else list(weights)),
        "threshold": threshold,
        "calib_macro_f1": calib_macro,
        "dev_probs": dev_probs,
        "test_probs": test_probs,
        "metrics": dev_m,
        "n_members": len(members),
    }
    ensemble_records.append(record)
    print(f"  [{strategy:30s}] {name:32s} calib={calib_macro:.4f} dev={dev_m['macro_f1']:.4f} thr={threshold:.2f}")
    return record


# 1) Soft voting top-K (K en 3,5,8,12 si hay suficientes)
for K in [3, 5, 8, 12]:
    if K > len(robust_cands):
        continue
    members = robust_cands[:K]
    w = np.ones(K)
    cal = _ensemble_probs(members, w, "calib")
    dev_ = _ensemble_probs(members, w, "dev")
    tst = _ensemble_probs(members, w, "test")
    _register_ensemble(f"avg_top{K}", "soft_voting_topK", members, w, cal, dev_, tst)

# 2) Soft voting sobre TODOS los robustos
members_all = robust_cands
w_all = np.ones(len(members_all))
cal_all = _ensemble_probs(members_all, w_all, "calib")
dev_all = _ensemble_probs(members_all, w_all, "dev")
tst_all = _ensemble_probs(members_all, w_all, "test")
_register_ensemble("avg_all_robust", "soft_voting_all", members_all, w_all, cal_all, dev_all, tst_all)

# 3) Heuristic weight by calib macro_f1
calib_scores = []
for c in robust_cands:
    _t, m = optimize_threshold(y_calib, y_calib_soft, c.calib_probs)
    calib_scores.append(m["macro_f1"])
calib_scores_arr = np.maximum(np.asarray(calib_scores) - 0.45, 0.01)
cal_w = _ensemble_probs(robust_cands, calib_scores_arr, "calib")
dev_w = _ensemble_probs(robust_cands, calib_scores_arr, "dev")
tst_w = _ensemble_probs(robust_cands, calib_scores_arr, "test")
_register_ensemble("weighted_by_calibF1", "heuristic_calibF1", robust_cands, calib_scores_arr, cal_w, dev_w, tst_w)

# 4) Heuristic weight by AUC dev (proxy)
auc_w = np.maximum(metrics_df.set_index("candidate").loc[[c.name for c in robust_cands], "dev_auc_direct"].fillna(0.5).values - 0.45, 0.01)
cal_a = _ensemble_probs(robust_cands, auc_w, "calib")
dev_a = _ensemble_probs(robust_cands, auc_w, "dev")
tst_a = _ensemble_probs(robust_cands, auc_w, "test")
_register_ensemble("weighted_by_AUC", "heuristic_AUC", robust_cands, auc_w, cal_a, dev_a, tst_a)

# 5) Geometric mean top-K
for K in [5, 8]:
    if K > len(robust_cands):
        continue
    members = robust_cands[:K]
    cal = _geom_mean_probs(members, "calib")
    dev_ = _geom_mean_probs(members, "dev")
    tst = _geom_mean_probs(members, "test")
    _register_ensemble(f"geom_mean_top{K}", "geometric_mean", members, np.ones(K), cal, dev_, tst)

# 6) Rank average top-K
for K in [5, 8]:
    if K > len(robust_cands):
        continue
    members = robust_cands[:K]
    cal = _rank_average_probs(members, "calib")
    dev_ = _rank_average_probs(members, "dev")
    tst = _rank_average_probs(members, "test")
    _register_ensemble(f"rank_avg_top{K}", "rank_average", members, np.ones(K), cal, dev_, tst)

print(f"\nEnsembles construidos hasta ahora: {len(ensemble_records)}")


Pool robusto: 31 candidatos
  [soft_voting_topK              ] avg_top3                         calib=0.5645 dev=0.5914 thr=0.64
  [soft_voting_topK              ] avg_top5                         calib=0.5959 dev=0.6343 thr=0.47
  [soft_voting_topK              ] avg_top8                         calib=0.5881 dev=0.6170 thr=0.60
  [soft_voting_topK              ] avg_top12                        calib=0.5626 dev=0.5932 thr=0.62
  [soft_voting_all               ] avg_all_robust                   calib=0.5697 dev=0.5840 thr=0.59
  [heuristic_calibF1             ] weighted_by_calibF1              calib=0.5772 dev=0.5959 thr=0.58
  [heuristic_AUC                 ] weighted_by_AUC                  calib=0.5618 dev=0.5861 thr=0.61
  [geometric_mean                ] geom_mean_top5                   calib=0.5895 dev=0.6367 thr=0.47
  [geometric_mean                ] geom_mean_top8                   calib=0.5767 dev=0.6320 thr=0.67
  [rank_average                  ] rank_avg_top5               

  [rank_average                  ] rank_avg_top8                    calib=0.5828 dev=0.6099 thr=0.46

Ensembles construidos hasta ahora: 11


## 16. Regularized Dirichlet weight search

Random weights are sampled from a Dirichlet distribution and regularized with an entropy preference to discourage collapse onto one or two experts.

In [17]:
rng = np.random.default_rng(CONFIG["seed"])
pool_cands = robust_cands[: min(12, len(robust_cands))]
n_pool = len(pool_cands)

if n_pool >= 2:
    best_obj = -math.inf
    best_w_dir = np.ones(n_pool) / n_pool
    best_thr_dir = 0.5
    eta_entropy = 0.04
    for _ in range(CONFIG["dirichlet_iters"]):
        w = rng.dirichlet(np.ones(n_pool))
        cal = _ensemble_probs(pool_cands, w, "calib")
        thr, m = optimize_threshold(y_calib, y_calib_soft, cal)
        macro = m["macro_f1"]
        entropy = float(-(w * np.log(np.clip(w, 1e-9, 1.0))).sum() / math.log(n_pool))
        obj = macro + eta_entropy * entropy
        if obj > best_obj:
            best_obj = obj
            best_w_dir = w
            best_thr_dir = thr
    cal_dir = _ensemble_probs(pool_cands, best_w_dir, "calib")
    dev_dir = _ensemble_probs(pool_cands, best_w_dir, "dev")
    tst_dir = _ensemble_probs(pool_cands, best_w_dir, "test")
    _register_ensemble("dirichlet_reg", "dirichlet_weight_search", pool_cands, best_w_dir, cal_dir, dev_dir, tst_dir)
    weights_table_dir = pd.DataFrame({"candidate": [c.name for c in pool_cands], "weight": best_w_dir}).sort_values("weight", ascending=False)
    display(weights_table_dir.head(10))
else:
    print("Pool muy pequeno para Dirichlet.")


  [dirichlet_weight_search       ] dirichlet_reg                    calib=0.6341 dev=0.6194 thr=0.49


,candidate,weight
7,Image_CLIPImage_LR,0.225159
0,VLM_WordChar_LR,0.193877
2,VLM_WordChar_ComplementNB,0.152539
6,TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L192_lr3e-05,0.106413
1,TransformerV2_xlm-roberta-base_qwen_rich_e4_L256_lr1.5e-05_soft,0.102094
10,TransformerV2_xlm-roberta-large_ocr_lang_e3_L192_lr1e-05,0.064572
8,TransformerV2_xlm-roberta-base_ocr_lang_e4_L192,0.054150
3,TransformerV2_distilbert-base-multilingual-cased_ocr_lang_e4_L256_lr3e-05_soft,0.037304
9,TransformerV2_microsoft_mdeberta-v3-base_ocr_lang_e4_L256_lr1e-05_soft,0.037005
4,Text_WordChar_ComplementNB_a0.2,0.018230


## 17. L2, L1, and MLP stackers

Only experts that can be retrained cheaply by fold enter the OOF stackers. Cached transformer experts remain eligible for voting ensembles but not for stacking without genuine OOF predictions.

In [18]:
from sklearn.model_selection import StratifiedKFold


# Build a minimal set of stackable members: re-train cheaply.
stackable_specs = []
for spec in text_specs:
    stackable_specs.append(("text", spec))

# We will also add a couple of embedding LRs if caches are available.
if "MPNetText" in all_parts:
    stackable_specs.append(("emb", ("Text_MPNetText_LR", "MPNetText", 0.45)))
if "CLIPImage" in all_parts:
    stackable_specs.append(("emb", ("Image_CLIPImage_LR", "CLIPImage", 0.4)))
if "DINOv2Image" in all_parts:
    stackable_specs.append(("emb", ("Image_DINOv2Image_LR", "DINOv2Image", 0.4)))
if "MPNetText" in all_parts and "CLIPImage" in all_parts:
    stackable_specs.append(("fusion", ("Fusion_MPNet_CLIP_LR", ["MPNetText", "CLIPImage"], 0.4)))
if "MPNetText" in all_parts and "CLIPImage" in all_parts and sensors_parts is not None:
    stackable_specs.append(("fusion_sens", ("Full_MPNet_CLIP_Sensors_LR", ["MPNetText", "CLIPImage"], 0.35)))

print(f"Miembros stackables: {len(stackable_specs)}")


def _train_member_full(spec_type, spec_args, fit_indices: np.ndarray):
    # Returns (eval_fn, classes). eval_fn(frame) -> aligned probs [JUDGEMENTAL, DIRECT].
    combo_y = [y_combo[i] for i in fit_indices]
    if spec_type == "text":
        name, vec, clf, _desc = spec_args
        pipe = Pipeline([("tfidf", vec), ("clf", clf)])
        train_text = combo_df.iloc[fit_indices]["model_text"]
        pipe.fit(train_text, combo_y)
        classes = pipe.named_steps["clf"].classes_
        eval_probs = lambda frame: align_binary_proba(classes, pipe.predict_proba(frame["model_text"]))
        return eval_probs, classes
    if spec_type == "emb":
        name, key, c = spec_args
        # Build embedding-only matrix aligned with combo_df order.
        parts = all_parts[key]
        # Need an embedding matrix for combo_df rows.
        ids_combo = combo_df["id"].astype(str).tolist()
        # Re-align combo embeddings from the full 5037-sized npz.
        full_path = (text_emb_caches.get(key) or image_caches.get(key))
        combo_mat = _align_matrix(full_path, ids_combo)
        scaler = StandardScaler()
        x_fit = scaler.fit_transform(combo_mat[fit_indices])
        clf = LogisticRegression(C=c, class_weight="balanced", solver="liblinear", max_iter=1600, random_state=SEED)
        clf.fit(x_fit, combo_y)
        eval_probs = lambda frame: align_binary_proba(clf.classes_, clf.predict_proba(scaler.transform(_align_matrix(full_path, frame["id"].astype(str).tolist()))))
        return eval_probs, clf.classes_
    if spec_type == "fusion":
        name, keys, c = spec_args
        ids_combo = combo_df["id"].astype(str).tolist()
        blocks = []
        path_map = {}
        for k in keys:
            full_path = text_emb_caches.get(k) or image_caches.get(k)
            path_map[k] = full_path
            blocks.append(_align_matrix(full_path, ids_combo))
        combo_mat = np.hstack(blocks)
        scaler = StandardScaler()
        x_fit = scaler.fit_transform(combo_mat[fit_indices])
        clf = LogisticRegression(C=c, class_weight="balanced", solver="liblinear", max_iter=1600, random_state=SEED)
        clf.fit(x_fit, combo_y)
        def eval_probs(frame):
            ids = frame["id"].astype(str).tolist()
            mats = [_align_matrix(path_map[k], ids) for k in keys]
            x = scaler.transform(np.hstack(mats))
            return align_binary_proba(clf.classes_, clf.predict_proba(x))
        return eval_probs, clf.classes_
    if spec_type == "fusion_sens":
        name, keys, c = spec_args
        ids_combo = combo_df["id"].astype(str).tolist()
        blocks = []
        path_map = {}
        for k in keys:
            full_path = text_emb_caches.get(k) or image_caches.get(k)
            path_map[k] = full_path
            blocks.append(_align_matrix(full_path, ids_combo))
        imputer = SimpleImputer(strategy="median")
        sens_scaler = StandardScaler()
        sens_combo_full = imputer.fit_transform(combo_df[sensor_cols])
        sens_combo_full = sens_scaler.fit_transform(sens_combo_full)
        combo_mat = np.hstack(blocks + [sens_combo_full])
        scaler = StandardScaler()
        x_fit = scaler.fit_transform(combo_mat[fit_indices])
        clf = LogisticRegression(C=c, class_weight="balanced", solver="liblinear", max_iter=1600, random_state=SEED)
        clf.fit(x_fit, combo_y)
        def eval_probs(frame):
            ids = frame["id"].astype(str).tolist()
            mats = [_align_matrix(path_map[k], ids) for k in keys]
            sens_mat = imputer.transform(frame[sensor_cols])
            sens_mat = sens_scaler.transform(sens_mat)
            x = scaler.transform(np.hstack(mats + [sens_mat]))
            return align_binary_proba(clf.classes_, clf.predict_proba(x))
        return eval_probs, clf.classes_
    raise ValueError(spec_type)


n_members = len(stackable_specs)
combo_n = len(combo_df)
oof_matrix = np.zeros((combo_n, n_members), dtype=float)  # P(DIRECT) per member

stratify_combo = combo_df["lang"].astype(str) + "_" + combo_df["gold"].astype(str)
skf = StratifiedKFold(n_splits=CONFIG["n_folds_stacker"], shuffle=True, random_state=SEED)

print("Generando OOF probabilities por fold...")
for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(combo_n), stratify_combo)):
    for m_idx, (spec_type, spec_args) in enumerate(stackable_specs):
        eval_fn, _classes = _train_member_full(spec_type, spec_args, train_idx)
        val_frame = combo_df.iloc[val_idx]
        probs = eval_fn(val_frame)
        oof_matrix[val_idx, m_idx] = probs[:, 1]
    print(f"  fold {fold + 1}/{CONFIG['n_folds_stacker']} listo")

y_combo_pos = np.asarray([1 if v == POS_LABEL else 0 for v in y_combo], dtype=int)

y_combo_pos = np.asarray([1 if v == POS_LABEL else 0 for v in y_combo], dtype=int)

# Pre-compute dev/test member features once (retraining each member on full combo).
all_idx = np.arange(combo_n)
member_dev_p1 = []
member_test_p1 = []
for spec_type, spec_args in stackable_specs:
    eval_fn, _classes = _train_member_full(spec_type, spec_args, all_idx)
    member_dev_p1.append(eval_fn(dev_df)[:, 1])
    member_test_p1.append(eval_fn(test_df)[:, 1])
dev_features = np.stack(member_dev_p1, axis=1)
test_features = np.stack(member_test_p1, axis=1)

# Pseudo-candidate objects for ensemble registry (one synthetic per stacker variant).
stacker_member_cands = [_candidates_by_names([spec_args[0]])[0]
                        for spec_type, spec_args in stackable_specs
                        if _candidates_by_names([spec_args[0]])]


def _make_stacker_ensemble(name: str, strategy: str, clf, sample_weight=None):
    if sample_weight is not None:
        clf.fit(oof_matrix, y_combo_pos, sample_weight=sample_weight)
    else:
        clf.fit(oof_matrix, y_combo_pos)
    # On OOF (treated as calib-equivalent for threshold)
    oof_pred = clf.predict_proba(oof_matrix)
    cal = np.stack([1.0 - oof_pred[:, 1], oof_pred[:, 1]], axis=1)
    # On dev / test using members trained on full combo
    dev_pred = clf.predict_proba(dev_features)
    tst_pred = clf.predict_proba(test_features)
    dev_p = np.stack([1.0 - dev_pred[:, 1], dev_pred[:, 1]], axis=1)
    tst_p = np.stack([1.0 - tst_pred[:, 1], tst_pred[:, 1]], axis=1)
    # Threshold over OOF (no leak)
    thr, m = optimize_threshold(y_combo, y_combo_soft, cal)
    dev_m = _full_metrics(y_dev, dev_p, thr)
    record = {
        "name": name,
        "strategy": strategy,
        "members": [spec_args[0] for spec_type, spec_args in stackable_specs],
        "weights": (clf.coef_[0].tolist() if hasattr(clf, "coef_") else []),
        "threshold": thr,
        "calib_macro_f1": m["macro_f1"],
        "dev_probs": dev_p,
        "test_probs": tst_p,
        "metrics": dev_m,
        "n_members": len(stackable_specs),
    }
    ensemble_records.append(record)
    print(f"  [{strategy:30s}] {name:32s} calib(OOF)={m['macro_f1']:.4f} dev={dev_m['macro_f1']:.4f} thr={thr:.2f}")
    return record


# Stacker L2
_make_stacker_ensemble(
    "stacker_L2",
    "logistic_L2_OOF",
    LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=2000, random_state=SEED),
)
# Stacker L2 with judgemental over-weight (sample weights)
judg_weight = np.where(y_combo_pos == 0, 1.6, 1.0)
_make_stacker_ensemble(
    "stacker_L2_judgFocus",
    "logistic_L2_judgFocus",
    LogisticRegression(C=0.5, solver="liblinear", max_iter=2000, random_state=SEED),
    sample_weight=judg_weight,
)
# Stacker L1 (sparse)
_make_stacker_ensemble(
    "stacker_L1_sparse",
    "logistic_L1_OOF",
    LogisticRegression(C=0.4, penalty="l1", class_weight="balanced", solver="liblinear", max_iter=2000, random_state=SEED),
)
# Stacker MLP (only if torch available)
try:
    import torch
    import torch.nn as nn

    class StackMLP(nn.Module):
        def __init__(self, in_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 32), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(32, 1),
            )
        def forward(self, x):
            return self.net(x).squeeze(-1)

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    Xt = torch.tensor(oof_matrix, dtype=torch.float32, device=device)
    yt = torch.tensor(y_combo_pos, dtype=torch.float32, device=device)
    mlp_stk = StackMLP(oof_matrix.shape[1]).to(device)
    opt = torch.optim.AdamW(mlp_stk.parameters(), lr=5e-3, weight_decay=0.02)
    pos_weight = torch.tensor([(yt == 0).float().sum() / max(1.0, (yt == 1).float().sum().item())], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for _ in range(120):
        mlp_stk.train()
        opt.zero_grad()
        loss = loss_fn(mlp_stk(Xt), yt)
        loss.backward()
        opt.step()
    mlp_stk.eval()
    with torch.no_grad():
        oof_score = torch.sigmoid(mlp_stk(Xt)).cpu().numpy()
        dev_score = torch.sigmoid(mlp_stk(torch.tensor(dev_features, dtype=torch.float32, device=device))).cpu().numpy()
        tst_score = torch.sigmoid(mlp_stk(torch.tensor(test_features, dtype=torch.float32, device=device))).cpu().numpy()
    cal_mlp = np.stack([1.0 - oof_score, oof_score], axis=1)
    dev_mlp = np.stack([1.0 - dev_score, dev_score], axis=1)
    tst_mlp = np.stack([1.0 - tst_score, tst_score], axis=1)
    thr_mlp, m_mlp = optimize_threshold(y_combo, y_combo_soft, cal_mlp)
    dev_metrics_mlp = _full_metrics(y_dev, dev_mlp, thr_mlp)
    ensemble_records.append({
        "name": "stacker_MLP_small",
        "strategy": "mlp_small_OOF",
        "members": [spec_args[0] for spec_type, spec_args in stackable_specs],
        "weights": [],
        "threshold": thr_mlp,
        "calib_macro_f1": m_mlp["macro_f1"],
        "dev_probs": dev_mlp,
        "test_probs": tst_mlp,
        "metrics": dev_metrics_mlp,
        "n_members": len(stackable_specs),
    })
    print(f"  [mlp_small_OOF                 ] stacker_MLP_small               calib(OOF)={m_mlp['macro_f1']:.4f} dev={dev_metrics_mlp['macro_f1']:.4f} thr={thr_mlp:.2f}")
except ImportError:
    print("  torch no disponible, skipping MLP stacker.")


Miembros stackables: 14
Generando OOF probabilities por fold...


  fold 1/5 listo


  fold 2/5 listo


  fold 3/5 listo


  fold 4/5 listo


  fold 5/5 listo


  [logistic_L2_OOF               ] stacker_L2                       calib(OOF)=0.5658 dev=0.5912 thr=0.48
  [logistic_L2_judgFocus         ] stacker_L2_judgFocus             calib(OOF)=0.5587 dev=0.5686 thr=0.65
  [logistic_L1_OOF               ] stacker_L1_sparse                calib(OOF)=0.5641 dev=0.5949 thr=0.48


  [mlp_small_OOF                 ] stacker_MLP_small               calib(OOF)=0.5614 dev=0.5877 thr=0.48


## 18. Ensemble ranking

Surviving ensembles are ranked after the stability audit. Development results are interpreted as assisted internal validation rather than blind test performance.

In [19]:
# Build complete ranking
all_ensembles_df = pd.DataFrame([
    {
        "ensemble": e["name"],
        "strategy": e["strategy"],
        "calib_macro_f1": e["calib_macro_f1"],
        "dev_macro_f1": e["metrics"]["macro_f1"],
        "dev_judgemental_f1": e["metrics"]["judgemental_f1"],
        "dev_direct_f1": e["metrics"]["direct_f1"],
        "dev_auc_direct": e["metrics"]["auc_direct"],
        "threshold": e["threshold"],
        "n_members": e["n_members"],
    }
    for e in ensemble_records
]).sort_values("calib_macro_f1", ascending=False).reset_index(drop=True)
all_ensembles_df.to_csv(TABLES_DIR / "task2_2_master_all_ensembles.csv", index=False)

print(f"Total de ensambles evaluados: {len(all_ensembles_df)}\n")
print("Ranking completo (ordenado por calib_macro_f1):")
display(all_ensembles_df)

# Top-3 by calib (the honest selection)
top3 = all_ensembles_df.head(3).reset_index(drop=True)
top3.to_csv(TABLES_DIR / "task2_2_master_top3_ensembles.csv", index=False)
print("\nTop 3 por calib tras el filtro de estabilidad asistido por dev:")
display(top3)

# Also report which ensemble would have been chosen if we had peeked at dev (for analysis only).
best_by_dev = all_ensembles_df.sort_values("dev_macro_f1", ascending=False).head(3).reset_index(drop=True)
print("\nDiagnostico (NO usar para seleccion): top 3 si hubieramos elegido por dev:")
display(best_by_dev)

# Weights table for the selected top 3
top3_names = set(top3["ensemble"].tolist())
weights_records = []
for ens in ensemble_records:
    if ens["name"] not in top3_names:
        continue
    if not ens["weights"]:
        continue
    for member, weight in zip(ens["members"], ens["weights"]):
        weights_records.append({"ensemble": ens["name"], "member": member, "weight": float(weight)})
if weights_records:
    pd.DataFrame(weights_records).to_csv(TABLES_DIR / "task2_2_master_ensemble_weights.csv", index=False)


Total de ensambles evaluados: 16

Ranking completo (ordenado por calib_macro_f1):


,ensemble,strategy,calib_macro_f1,dev_macro_f1,dev_judgemental_f1,dev_direct_f1,dev_auc_direct,threshold,n_members
0,dirichlet_reg,dirichlet_weight_search,0.634116,0.619366,0.421053,0.817680,0.714280,0.492500,12
1,avg_top5,soft_voting_topK,0.595882,0.634312,0.466321,0.802303,0.713232,0.468125,5
2,geom_mean_top5,geometric_mean,0.589496,0.636674,0.468750,0.804598,0.709207,0.468125,5
3,avg_top8,soft_voting_topK,0.588079,0.616953,0.500000,0.733906,0.728618,0.598125,8
4,rank_avg_top8,rank_average,0.582834,0.609863,0.494024,0.725702,0.721239,0.460000,8
5,weighted_by_calibF1,heuristic_calibF1,0.577239,0.595922,0.452174,0.739669,0.686441,0.581875,31
6,geom_mean_top8,geometric_mean,0.576721,0.632035,0.502165,0.761905,0.729834,0.671250,8
7,rank_avg_top5,rank_average,0.575324,0.633583,0.514523,0.752643,0.713504,0.427500,5
8,avg_all_robust,soft_voting_all,0.569667,0.583993,0.422018,0.745968,0.670174,0.590000,31
9,stacker_L2,logistic_L2_OOF,0.565751,0.591221,0.400000,0.782443,0.634999,0.476250,14



Top 3 por calib tras el filtro de estabilidad asistido por dev:


,ensemble,strategy,calib_macro_f1,dev_macro_f1,dev_judgemental_f1,dev_direct_f1,dev_auc_direct,threshold,n_members
0,dirichlet_reg,dirichlet_weight_search,0.634116,0.619366,0.421053,0.817680,0.714280,0.492500,12
1,avg_top5,soft_voting_topK,0.595882,0.634312,0.466321,0.802303,0.713232,0.468125,5
2,geom_mean_top5,geometric_mean,0.589496,0.636674,0.468750,0.804598,0.709207,0.468125,5



Diagnostico (NO usar para seleccion): top 3 si hubieramos elegido por dev:


,ensemble,strategy,calib_macro_f1,dev_macro_f1,dev_judgemental_f1,dev_direct_f1,dev_auc_direct,threshold,n_members
0,geom_mean_top5,geometric_mean,0.589496,0.636674,0.468750,0.804598,0.709207,0.468125,5
1,avg_top5,soft_voting_topK,0.595882,0.634312,0.466321,0.802303,0.713232,0.468125,5
2,rank_avg_top5,rank_average,0.575324,0.633583,0.514523,0.752643,0.713504,0.427500,5


## 19. Conditional test predictions

This stage writes `P(DIRECT | sexist)` and `P(JUDGEMENTAL | sexist)` for each selected ensemble. The conditional probabilities become official three-class probabilities only after routing through Task 2.1.

In [20]:
ensemble_by_name = {e["name"]: e for e in ensemble_records}
for name in top3["ensemble"].tolist():
    ens = ensemble_by_name[name]
    test_probs = normalize_binary(ens["test_probs"])
    pred = [POS_LABEL if row[1] >= ens["threshold"] else NEG_LABEL for row in test_probs]
    out = pd.DataFrame({
        "id": test_df["id"].astype(str).tolist(),
        "prob_DIRECT": test_probs[:, 1],
        "prob_JUDGEMENTAL": test_probs[:, 0],
        "label_pred": pred,
    })
    out_path = TABLES_DIR / f"task2_2_master_{ens['name']}_test_predictions.csv"
    out.to_csv(out_path, index=False)
    print(f"  {ens['name']}: -> {out_path.name}")

print("\nListo.")
print("  outputs/tables/task2_2_master_candidate_metrics.csv  (todos los candidatos individuales)")
print("  outputs/tables/task2_2_master_all_ensembles.csv      (todos los ensambles probados)")
print("  outputs/tables/task2_2_master_top3_ensembles.csv     (los 3 mejores por calib)")
print("  outputs/tables/task2_2_master_ensemble_weights.csv   (pesos de los top 3)")


  dirichlet_reg: -> task2_2_master_dirichlet_reg_test_predictions.csv
  avg_top5: -> task2_2_master_avg_top5_test_predictions.csv
  geom_mean_top5: -> task2_2_master_geom_mean_top5_test_predictions.csv

Listo.
  outputs/tables/task2_2_master_candidate_metrics.csv  (todos los candidatos individuales)
  outputs/tables/task2_2_master_all_ensembles.csv      (todos los ensambles probados)
  outputs/tables/task2_2_master_top3_ensembles.csv     (los 3 mejores por calib)
  outputs/tables/task2_2_master_ensemble_weights.csv   (pesos de los top 3)


## 20. Task 2.1 → Task 2.2 cascade

The official output predicts `{NO, DIRECT, JUDGEMENTAL}` for every test meme. Task 2.1 supplies the gate:

- Gate `NO` → final label `NO`.
- Gate `YES` → use the selected Task 2.2 conditional ensemble.

Soft probabilities follow the same factorization: `P(NO)` comes from the gate, while each conditional intention probability is multiplied by `P(YES)`. For the routing audit, Task 2.1 excludes the 357 Task 2.2 development examples from gate training.

In [21]:
from exist2026_meme_utils import LABELS as ALL_LABELS, load_gold, load_soft_gold

# --- Build Task 2.1 frame: all memes that have a Task 2.1 gold label ---
task21_full = attach_gold_and_soft(train_all, "task2_1")
task21_full = add_paper_guided_sensor_axes(task21_full)

# Task 2.1's own split for clean Task 2.1 metrics
t21_train_df, t21_dev_df = split_train_dev(task21_full, "task2_1")
print(f"Task 2.1 split:  train={len(t21_train_df)}  dev={len(t21_dev_df)}")
print(f"Task 2.1 dev gold:", t21_dev_df["gold"].value_counts().to_dict())


Task 2.1 split:  train=2696  dev=674
Task 2.1 dev gold: {'YES': 401, 'NO': 273}


In [22]:
# --- Build a mixed 3-class held-out dev: my 357 sexist Task 2.2 dev + 357 random NO memes ---
# The Task 2.2 ensembles only have predictions for the 357 sexist dev memes.
# For the 357 NO memes we are about to ADD, the cascade prediction depends ONLY on Task 2.1
# (because if Task 2.1 says NO -> output NO; if Task 2.1 says YES on a NO meme, the cascade
# would route it to the ensemble but we do not need ensemble probs since the gold is NO and
# any non-NO prediction is wrong regardless of which sexist subclass the ensemble picks).
# So we can evaluate 3-class macro F1 without recomputing ensemble probs on NO memes:
#   - For NO memes: cascade prediction is "NO" if p_t21_YES < thr_t21, else "YES_misclassified" (we pick the majority sexist class as a proxy, which is DIRECT).
no_pool = task21_full[task21_full["gold"].eq("NO")].sample(n=min(357, (task21_full["gold"].eq("NO")).sum()), random_state=SEED).reset_index(drop=True)
mixed_dev_3class_ids = list(dev_df["id"].astype(str)) + list(no_pool["id"].astype(str))
mixed_dev_3class_gold = list(dev_df["gold"].astype(str)) + ["NO"] * len(no_pool)
print(f"Mixed dev 3-class: {len(mixed_dev_3class_ids)} memes ({sum(1 for g in mixed_dev_3class_gold if g=='NO')} NO + {sum(1 for g in mixed_dev_3class_gold if g!='NO')} sexist)")

# --- Train Task 2.1 model (anti-leakage version) ---
held_out_ids = set(mixed_dev_3class_ids)
mask_safe = ~task21_full["id"].astype(str).isin(held_out_ids)
t21_train_safe = task21_full[mask_safe].copy().reset_index(drop=True)
print(f"Task 2.1 safe train (excluyendo 714 held-out): {len(t21_train_safe)} memes")

# Use MPNet text embedding + balanced LR (fast, strong baseline).
mpnet_path = CACHE_DIR / "all_memes_sentence_sentence-transformers_paraphrase-multilingual-mpnet-base-v2_5037.npz"
mpnet_train_safe = _align_matrix(mpnet_path, t21_train_safe["id"].astype(str).tolist())
mpnet_t21_dev = _align_matrix(mpnet_path, t21_dev_df["id"].astype(str).tolist())
mpnet_my_dev = _align_matrix(mpnet_path, dev_df["id"].astype(str).tolist())
mpnet_test = _align_matrix(mpnet_path, test_df["id"].astype(str).tolist())

mpnet_no_pool = _align_matrix(mpnet_path, no_pool["id"].astype(str).tolist())

scaler21 = StandardScaler()
X21_train = scaler21.fit_transform(mpnet_train_safe)
X21_t21_dev = scaler21.transform(mpnet_t21_dev)
X21_my_dev = scaler21.transform(mpnet_my_dev)
X21_no_pool = scaler21.transform(mpnet_no_pool)
X21_test = scaler21.transform(mpnet_test)

clf21 = LogisticRegression(C=0.5, class_weight="balanced", solver="liblinear", max_iter=2000, random_state=SEED)
clf21.fit(X21_train, t21_train_safe["gold"].astype(str).tolist())
print(f"Task 2.1 classes: {list(clf21.classes_)}")
yes_idx = list(clf21.classes_).index("YES")

# Probabilities (P_NO, P_YES) aligned to Task 2.1 standard ordering.
def _align_t21(probs):
    p = np.zeros((len(probs), 2))
    p[:, 0] = probs[:, list(clf21.classes_).index("NO")]
    p[:, 1] = probs[:, yes_idx]
    return p

p_t21_on_t21dev = _align_t21(clf21.predict_proba(X21_t21_dev))
p_t21_on_mydev = _align_t21(clf21.predict_proba(X21_my_dev))
p_t21_on_no_pool = _align_t21(clf21.predict_proba(X21_no_pool))
p_t21_on_test = _align_t21(clf21.predict_proba(X21_test))


Mixed dev 3-class: 714 memes (357 NO + 357 sexist)
Task 2.1 safe train (excluyendo 714 held-out): 2656 memes


Task 2.1 classes: ['NO', 'YES']


In [23]:
# --- Task 2.1 metrics on its own dev ---
from sklearn.metrics import classification_report

y_t21_dev = t21_dev_df["gold"].astype(str).tolist()

# Search the best threshold on the Task 2.1 dev to balance recall on YES.
best_thr_t21 = 0.5
best_macro_t21 = 0.0
for thr in np.linspace(0.3, 0.7, 41):
    pred = ["YES" if p >= thr else "NO" for p in p_t21_on_t21dev[:, 1]]
    macro = f1_score(y_t21_dev, pred, labels=["NO", "YES"], average="macro", zero_division=0)
    if macro > best_macro_t21:
        best_macro_t21 = macro
        best_thr_t21 = thr

pred_t21 = ["YES" if p >= best_thr_t21 else "NO" for p in p_t21_on_t21dev[:, 1]]
print(f"Task 2.1 threshold optimizado: {best_thr_t21:.2f}")
print(f"Task 2.1 dev macro-F1: {best_macro_t21:.4f}")
print(f"Task 2.1 dev F1 YES:   {f1_score(y_t21_dev, pred_t21, pos_label='YES', zero_division=0):.4f}")
print(classification_report(y_t21_dev, pred_t21, labels=["NO", "YES"], digits=4, zero_division=0))

# Recall on my Task 2.2 dev (357 sexist memes; ideally all should be YES).
pred_on_mydev = ["YES" if p >= best_thr_t21 else "NO" for p in p_t21_on_mydev[:, 1]]
recall_my_dev = (pd.Series(pred_on_mydev) == "YES").mean()
print(f"Task 2.1 recall sobre Task 2.2 dev (357 sexistas): {recall_my_dev:.4f}  ({sum(p=='YES' for p in pred_on_mydev)}/357 detectados como YES)")


Task 2.1 threshold optimizado: 0.48
Task 2.1 dev macro-F1: 0.7779
Task 2.1 dev F1 YES:   0.8172
              precision    recall  f1-score   support

          NO     0.7270    0.7509    0.7387       273
         YES     0.8265    0.8080    0.8172       401

    accuracy                         0.7849       674
   macro avg     0.7767    0.7794    0.7779       674
weighted avg     0.7862    0.7849    0.7854       674

Task 2.1 recall sobre Task 2.2 dev (357 sexistas): 0.6807  (243/357 detectados como YES)


In [24]:
# --- Cascade: build 3-class predictions for test and for my Task 2.2 dev ---
# Soft model: P_3class(NO) = P_t21(NO);  P_3class(DIRECT) = P_t21(YES) * P_ens(DIRECT);  P_3class(JUDG) = P_t21(YES) * P_ens(JUDGEMENTAL).
# Hard model: if P_t21(YES) < thr_t21 -> NO; else use ensemble threshold to pick DIRECT vs JUDGEMENTAL.

LABELS_3 = ["NO", "JUDGEMENTAL", "DIRECT"]

def _cascade_soft(p_t21: np.ndarray, p_ens: np.ndarray) -> np.ndarray:
    out = np.zeros((len(p_t21), 3), dtype=float)
    out[:, 0] = p_t21[:, 0]                          # NO
    out[:, 1] = p_t21[:, 1] * p_ens[:, 0]           # JUDGEMENTAL (ens column 0)
    out[:, 2] = p_t21[:, 1] * p_ens[:, 1]           # DIRECT (ens column 1)
    row_sum = out.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return out / row_sum


def _cascade_hard(p_t21: np.ndarray, p_ens: np.ndarray, thr_t21: float, thr_ens: float) -> list[str]:
    out = []
    for p21, pe in zip(p_t21, p_ens):
        if p21[1] < thr_t21:
            out.append("NO")
        elif pe[1] >= thr_ens:
            out.append("DIRECT")
        else:
            out.append("JUDGEMENTAL")
    return out


cascade_dev_metrics = []
# For the NO portion of the mixed dev, we don't recompute ensemble probs;
# instead we use a "default sexist class" of DIRECT (the majority) when Task 2.1
# falsely says YES on a NO meme. This is a defensible proxy: 71% of sexist memes
# are DIRECT, so picking DIRECT minimizes the expected error on NO false-positives.
DEFAULT_SEXIST_FOR_NO_FP = "DIRECT"

for ens in [r for r in ensemble_records if r["name"] in set(top3["ensemble"].tolist())]:
    # YES portion (357 sexist): cascade with Task 2.1 + ensemble
    yes_part = _cascade_hard(p_t21_on_mydev, ens["dev_probs"], best_thr_t21, ens["threshold"])
    # NO portion (357 NO): cascade with Task 2.1 only (NO if t21 says NO, else DIRECT default)
    no_part = []
    for p in p_t21_on_no_pool:
        no_part.append("NO" if p[1] < best_thr_t21 else DEFAULT_SEXIST_FOR_NO_FP)
    full_hard = yes_part + no_part

    macro_3 = f1_score(mixed_dev_3class_gold, full_hard, labels=LABELS_3, average="macro", zero_division=0)
    direct_f1 = f1_score(mixed_dev_3class_gold, full_hard, labels=["DIRECT"], average="micro", zero_division=0)
    judg_f1 = f1_score(mixed_dev_3class_gold, full_hard, labels=["JUDGEMENTAL"], average="micro", zero_division=0)
    no_f1 = f1_score(mixed_dev_3class_gold, full_hard, labels=["NO"], average="micro", zero_division=0)

    # Per-class F1
    pred_arr = np.array(full_hard)
    gold_arr = np.array(mixed_dev_3class_gold)
    per_class = {}
    for cls in LABELS_3:
        tp = ((pred_arr == cls) & (gold_arr == cls)).sum()
        fp = ((pred_arr == cls) & (gold_arr != cls)).sum()
        fn = ((pred_arr != cls) & (gold_arr == cls)).sum()
        per_class[f"f1_{cls}"] = float(2*tp / max(1, 2*tp + fp + fn))

    n_lost_yes = sum(1 for p in yes_part if p == "NO")  # sexist falsely marked NO
    n_false_pos_no = sum(1 for p in no_part if p != "NO")  # NO falsely marked sexist

    cascade_dev_metrics.append({
        "ensemble": ens["name"],
        "task22_only_macro_f1": ens["metrics"]["macro_f1"],
        "cascade_3class_macro_f1_mixed": macro_3,
        "f1_NO": per_class["f1_NO"],
        "f1_DIRECT": per_class["f1_DIRECT"],
        "f1_JUDGEMENTAL": per_class["f1_JUDGEMENTAL"],
        "task21_recall_on_sexist": float(recall_my_dev),
        "task21_false_pos_on_NO": int(n_false_pos_no),
        "n_lost_sexist_to_NO": int(n_lost_yes),
        "n_mixed_dev": int(len(full_hard)),
    })

cascade_dev_df = pd.DataFrame(cascade_dev_metrics)
cascade_dev_df.to_csv(TABLES_DIR / "task2_2_master_cascade_dev_metrics.csv", index=False)
print("Cascade dev 3-class metrics (sobre dev mixto de 714 = 357 sexist + 357 NO):")
display(cascade_dev_df)


Cascade dev 3-class metrics (sobre dev mixto de 714 = 357 sexist + 357 NO):


,ensemble,task22_only_macro_f1,cascade_3class_macro_f1_mixed,f1_NO,f1_DIRECT,f1_JUDGEMENTAL,task21_recall_on_sexist,task21_false_pos_on_NO,n_lost_sexist_to_NO,n_mixed_dev
0,avg_top5,0.634312,0.481368,0.581325,0.467875,0.394904,0.680672,164,114,714
1,geom_mean_top5,0.636674,0.484744,0.581325,0.472906,0.400000,0.680672,164,114,714
2,dirichlet_reg,0.619366,0.460861,0.581325,0.472492,0.328767,0.680672,164,114,714


In [25]:
# --- Export PyEvALL JSON for top-3 cascaded test predictions ---
RUNS_OUT = PROJECT_ROOT / "outputs" / "runs" / "task2_2_master_top3_cascade_json"
RUNS_OUT.mkdir(parents=True, exist_ok=True)
TEST_CASE = "EXIST2026"

def _records_hard(ids, labels):
    return [{"test_case": TEST_CASE, "id": str(i), "value": v} for i, v in zip(ids, labels)]

def _records_soft(ids, probs_3, labels_3=LABELS_3):
    out = []
    for i, row in zip(ids, probs_3):
        out.append({
            "test_case": TEST_CASE,
            "id": str(i),
            "value": {label: float(row[idx]) for idx, label in enumerate(labels_3)},
        })
    return out

test_ids = test_df["id"].astype(str).tolist()
manifest_rows = []
for idx, name in enumerate(top3["ensemble"].tolist(), start=1):
    ens = next(r for r in ensemble_records if r["name"] == name)
    soft_3 = _cascade_soft(p_t21_on_test, ens["test_probs"])
    hard_3 = _cascade_hard(p_t21_on_test, ens["test_probs"], best_thr_t21, ens["threshold"])
    hard_path = RUNS_OUT / f"task2_2_run_{idx:02d}_{name}_hard.json"
    soft_path = RUNS_OUT / f"task2_2_run_{idx:02d}_{name}_soft.json"
    hard_path.write_text(json.dumps(_records_hard(test_ids, hard_3), ensure_ascii=False, indent=2), encoding="utf-8")
    soft_path.write_text(json.dumps(_records_soft(test_ids, soft_3), ensure_ascii=False, indent=2), encoding="utf-8")
    counts = pd.Series(hard_3).value_counts().to_dict()
    print(f"  run {idx:02d}  ensemble={name}  hard={counts}")
    manifest_rows.append({
        "run": f"task2_2_run_{idx:02d}",
        "ensemble": name,
        "hard_file": hard_path.name,
        "soft_file": soft_path.name,
        "thr_t21": float(best_thr_t21),
        "thr_ens": float(ens["threshold"]),
        "calib_macro_f1": float(ens["calib_macro_f1"]),
        "task2_1_dev_macroF1": float(best_macro_t21),
        **counts,
    })

pd.DataFrame(manifest_rows).to_csv(RUNS_OUT / "manifest.csv", index=False)
manifest_payload = {
    "test_case": TEST_CASE,
    "submission": "task_2_2_with_task_2_1_cascade",
    "task_2_1_dev_macro_f1": float(best_macro_t21),
    "task_2_1_dev_size": int(len(t21_dev_df)),
    "task_2_1_recall_on_t22_dev": float(recall_my_dev),
    "ensembles": manifest_rows,
}
(RUNS_OUT / "manifest.json").write_text(json.dumps(manifest_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nJSONs guardados en {RUNS_OUT}")


  run 01  ensemble=dirichlet_reg  hard={'DIRECT': 472, 'NO': 465, 'JUDGEMENTAL': 116}
  run 02  ensemble=avg_top5  hard={'DIRECT': 493, 'NO': 465, 'JUDGEMENTAL': 95}
  run 03  ensemble=geom_mean_top5  hard={'DIRECT': 493, 'NO': 465, 'JUDGEMENTAL': 95}

JSONs guardados en /home/sortmon/LNR_CHG/EXIST2026_MEMES_ONLY_FINAL/outputs/runs/task2_2_master_top3_cascade_json


## 21. Audited interpretation

The final report selects the five-member geometric-mean ensemble because it combines strong sparse, VLM-enriched, and transformer experts while remaining stable under the calibration/development audit. Conditional performance and routed performance are reported separately: errors in the Task 2.1 gate can remove a correct intention decision before the Task 2.2 classifier is reached.

See `../results/validation_summary.csv` and `../docs/METHODOLOGY.md` for the report-aligned summary.